# MIMIC-III BigQuery Cohort Extraction: Female ICU Patients with Blood Diseases

This Colab notebook runs a BigQuery SQL query against the MIMIC-III clinical dataset to create a model-ready CSV file for female ICU patients diagnosed with ICD-9 blood diseases codes 280–289.

The output dataset includes ICU stay information, binary mortality label, first-24-hour heart rate features, and first-24-hour laboratory features: hemoglobin, hematocrit, and glucose.


## 1. Install and Import Required Libraries

Colab usually includes the Google BigQuery client libraries, but the install cell below ensures the required packages are available.


In [1]:
# Install required Google Cloud BigQuery libraries
!pip -q install google-cloud-bigquery pandas pyarrow

# Import Python libraries
import pandas as pd
from google.colab import auth, files
from google.cloud import bigquery


## 2. Authenticate Google Cloud Access

Run the cell below and sign in with the Google account that has access to PhysioNet/MIMIC-III on BigQuery.


In [2]:
# Authenticate your Google account for BigQuery access
auth.authenticate_user()
print('Authenticated successfully.')


Authenticated successfully.


## 3. Steps to Execute the code block below:
Set Your Google Cloud Project for Big Query Client.

Use the following steps to get the project ID

1. Navigate to console.cloud.google.com
2. Copy the project ID and replace the value for the tag PROJECT_ID below.






In [9]:
# Replace with your Google Cloud project ID to execute the code
PROJECT_ID = 'project-4044d9a5-d0d2-4d71-b82'

# Create BigQuery client
client = bigquery.Client(project=PROJECT_ID)
print(f'BigQuery client created for project: {PROJECT_ID}')


BigQuery client created for project: project-4044d9a5-d0d2-4d71-b82


## 4. BigQuery SQL Query

This query extracts the female ICU blood-disease cohort and builds first-24-hour clinical features.

Important implementation notes:

- BigQuery table names are wrapped with backticks.
- The final `SELECT` explicitly selects feature columns to avoid duplicate `hadm_id` / `icustay_id` columns from `USING` joins.
- `itemid` filters are included inside the vitals/labs CTEs to reduce scanned data.


In [10]:
query = r'''
-- ============================================================
-- DESCRIPTION:
-- This query extracts a cohort of FEMALE ICU patients diagnosed
-- with BLOOD DISEASES (ICD-9: 280–289).
--
-- It builds a model-ready dataset including:
--   • ICU stay information
--   • Mortality label (binary)
--   • First 24-hour vital signs (heart rate)
--   • First 24-hour lab values (hemoglobin, hematocrit, glucose)
--
-- TABLES USED:
--   • diagnoses_icd   → disease identification
--   • icustays        → ICU admission details
--   • patients        → demographics (gender)
--   • admissions      → mortality (death time)
--   • chartevents     → vital signs (time-series)
--   • labevents       → laboratory measurements
-- ============================================================

WITH blood_disease AS (
  SELECT DISTINCT hadm_id
  FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
  WHERE icd9_code BETWEEN '280' AND '289'
),

icu_cohort AS (
  SELECT
         icu.subject_id,
         icu.hadm_id,
         icu.icustay_id,
         pat.gender,
         icu.intime,
         icu.outtime,
         adm.deathtime,
         CASE
           WHEN adm.deathtime IS NOT NULL THEN 1
           ELSE 0
         END AS mortality
  FROM `physionet-data.mimiciii_clinical.icustays` icu
  JOIN `physionet-data.mimiciii_clinical.patients` pat
       ON icu.subject_id = pat.subject_id
  JOIN `physionet-data.mimiciii_clinical.admissions` adm
       ON icu.hadm_id = adm.hadm_id
  JOIN blood_disease bd
       ON icu.hadm_id = bd.hadm_id
  WHERE pat.gender = 'F'
),

vitals_24h AS (
  SELECT
         ce.icustay_id,
         AVG(CASE WHEN ce.itemid IN (211, 220045) THEN ce.valuenum END) AS heart_rate_mean,
         MIN(CASE WHEN ce.itemid IN (211, 220045) THEN ce.valuenum END) AS heart_rate_min,
         MAX(CASE WHEN ce.itemid IN (211, 220045) THEN ce.valuenum END) AS heart_rate_max
  FROM `physionet-data.mimiciii_clinical.chartevents` ce
  JOIN icu_cohort icu
       ON ce.icustay_id = icu.icustay_id
  WHERE ce.charttime BETWEEN icu.intime
                         AND DATETIME_ADD(icu.intime, INTERVAL 24 HOUR)
    AND ce.itemid IN (211, 220045)
    AND ce.valuenum IS NOT NULL
  GROUP BY ce.icustay_id
),

labs_24h AS (
  SELECT
         le.hadm_id,
         AVG(CASE WHEN le.itemid = 50811 THEN le.valuenum END) AS hemoglobin,
         AVG(CASE WHEN le.itemid = 50813 THEN le.valuenum END) AS hematocrit,
         AVG(CASE WHEN le.itemid = 50809 THEN le.valuenum END) AS glucose
  FROM `physionet-data.mimiciii_clinical.labevents` le
  JOIN icu_cohort icu
       ON le.hadm_id = icu.hadm_id
  WHERE le.charttime BETWEEN icu.intime
                         AND DATETIME_ADD(icu.intime, INTERVAL 24 HOUR)
    AND le.itemid IN (50811, 50813, 50809)
    AND le.valuenum IS NOT NULL
  GROUP BY le.hadm_id
)

SELECT
       icu.*,
       vit.heart_rate_mean,
       vit.heart_rate_min,
       vit.heart_rate_max,
       lab.hemoglobin,
       lab.hematocrit,
       lab.glucose
FROM icu_cohort icu
LEFT JOIN vitals_24h vit
       USING(icustay_id)
LEFT JOIN labs_24h lab
       USING(hadm_id);
'''

print(query[:1000])



-- ============================================================
-- DESCRIPTION:
-- This query extracts a cohort of FEMALE ICU patients diagnosed
-- with BLOOD DISEASES (ICD-9: 280–289).
--
-- It builds a model-ready dataset including:
--   • ICU stay information
--   • Mortality label (binary)
--   • First 24-hour vital signs (heart rate)
--   • First 24-hour lab values (hemoglobin, hematocrit, glucose)
--
-- TABLES USED:
--   • diagnoses_icd   → disease identification
--   • icustays        → ICU admission details
--   • patients        → demographics (gender)
--   • admissions      → mortality (death time)
--   • chartevents     → vital signs (time-series)
--   • labevents       → laboratory measurements
-- ============================================================

WITH blood_disease AS (
  SELECT DISTINCT hadm_id
  FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
  WHERE icd9_code BETWEEN '280' AND '289'
),

icu_cohort AS (
  SELECT 
         icu.subject_id,
         icu.ha

## 5. Run the Query and Load Results into a DataFrame

This may take some time depending on BigQuery queue time and the amount of data scanned.


In [11]:
# Run BigQuery SQL and convert the result to a pandas DataFrame
query_job = client.query(query)
df = query_job.result().to_dataframe()

print('Query completed.')
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.head()


Query completed.
Rows: 9747
Columns: 14


,subject_id,hadm_id,icustay_id,gender,intime,outtime,deathtime,mortality,heart_rate_mean,heart_rate_min,heart_rate_max,hemoglobin,hematocrit,glucose
0,27400,141242,243307,F,2118-08-31 20:43:00,2118-09-05 18:43:44,NaT,0,134.480000,121.0,148.0,NaN,NaN,NaN
1,28941,107962,243236,F,2144-07-31 14:55:49,2144-08-07 18:28:40,NaT,0,91.075000,44.0,145.0,10.0,1.50,NaN
2,29054,191001,291245,F,2147-02-06 17:20:27,2147-02-07 22:20:44,NaT,0,119.166667,107.0,135.0,NaN,NaN,NaN
3,32500,172723,282079,F,2138-04-29 18:51:41,2138-05-16 16:44:13,NaT,0,117.238095,107.0,124.0,14.2,2.35,NaN
4,346,186516,258237,F,2148-11-30 19:16:19,2148-12-05 18:30:53,NaT,0,76.061224,39.0,105.0,NaN,0.70,NaN


## 6. Quick Dataset Checks

These checks confirm the cohort size, mortality distribution, missing values, and basic numeric summaries.


In [12]:
# Display basic information about the extracted cohort
print('Dataset shape:', df.shape)

print('\nMortality counts:')
print(df['mortality'].value_counts(dropna=False))

print('\nMortality rate:')
print(df['mortality'].mean())

print('\nMissing values per column:')
print(df.isna().sum())

print('\nNumeric summary:')
df.describe(include='all')


Dataset shape: (9747, 14)

Mortality counts:
mortality
0    8462
1    1285
Name: count, dtype: Int64

Mortality rate:
0.13183543654457783

Missing values per column:
subject_id            0
hadm_id               0
icustay_id            0
gender                0
intime                0
outtime               0
deathtime          8462
mortality             0
heart_rate_mean     197
heart_rate_min      197
heart_rate_max      197
hemoglobin         7871
hematocrit         4843
glucose            7330
dtype: int64

Numeric summary:


,subject_id,hadm_id,icustay_id,gender,intime,outtime,deathtime,mortality,heart_rate_mean,heart_rate_min,heart_rate_max,hemoglobin,hematocrit,glucose
count,9747.0,9747.0,9747.0,9747,9747,9747,1285,9747.0,9550.000000,9550.000000,9550.000000,1876.000000,4904.000000,2417.000000
unique,<NA>,<NA>,<NA>,1,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
top,<NA>,<NA>,<NA>,F,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
freq,<NA>,<NA>,<NA>,9747,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
mean,38995.780548,149802.409152,249821.384426,NaN,2151-05-20 14:19:46.058992,2151-05-25 06:33:04.541603,2151-10-26 07:15:56.544747,0.131835,87.735853,72.959199,106.348963,9.498295,2.313916,147.567703
min,6.0,100028.0,200009.0,NaN,2100-07-02 01:12:19,2100-07-05 21:38:49,2100-09-11 14:21:00,0.0,34.700000,0.000000,39.000000,0.000000,0.300000,23.000000
25%,14467.0,125300.0,225079.0,NaN,2126-05-22 20:09:42,2126-05-30 18:35:52,2128-01-08 03:32:00,0.0,76.121250,63.000000,91.000000,8.225000,1.200000,114.000000
50%,28712.0,149957.0,249950.0,NaN,2150-12-10 12:51:57,2150-12-18 11:25:26,2151-08-09 19:00:00,0.0,86.732051,72.000000,104.000000,9.425000,1.800000,132.625000
75%,63801.0,174746.5,274567.5,NaN,2176-08-15 03:53:50,2176-08-19 07:25:52,2176-03-19 16:25:00,0.0,98.362727,83.000000,120.000000,10.600000,2.602500,159.500000
max,99995.0,199992.0,299994.0,NaN,2208-08-19 13:03:37,2208-08-21 05:38:09,2208-02-05 11:45:00,1.0,160.958333,154.000000,950.000000,18.233333,24.300000,946.000000


## 7. Save Output as CSV

The cell below saves the BigQuery output as a CSV file inside the Colab runtime.


In [13]:
# Save output dataset to CSV
output_file = 'female_blood_disease_icu.csv'
df.to_csv(output_file, index=False)

print(f'Saved CSV file: {output_file}')


Saved CSV file: female_blood_disease_icu.csv


## 8. Download CSV File to Your Computer

Run the cell below to download the CSV output file from Colab.


In [14]:
# Download CSV file from Colab to your local machine
files.download(output_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: Save CSV to Google Drive

Use this if you want to keep the output file in Google Drive instead of only downloading it locally.


In [ ]:
# Optional: Save output to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# drive_output_path = '/content/drive/MyDrive/female_icu_blood_disease_mimiciii_24h_features.csv'
# df.to_csv(drive_output_path, index=False)
# print(f'Saved to Google Drive: {drive_output_path}')
